# PERSUADE Score Long Analysis (v3)

Clean analysis of **long essays only** (all >= 778 tokens) for score comparison.

**Key Features:**
- All 150 essays use EXTENDED regime (full window coverage)
- Primary DV: `half_life` (full, not half_life_128) - comparable since all essays have W=512
- Secondary DVs: `delta_max`, `delta_128`, `delta_late` (= delta_max - delta_128)

**Cohort:** `persuade_score_long_cohort.jsonl` (50 low, 50 mid, 50 high)

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes pandas numpy matplotlib seaborn tqdm statsmodels scipy

In [ ]:
import json
import math
import os
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION - SCORE_LONG SPECIFIC
# ============================================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive/LRTIA/Data/persuade_clean"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade"
COHORT_PATH = f"{DRIVE_BASE}/cohorts/persuade_score_long_cohort.jsonl"

EXPERIMENT = 'score_long'

# Model
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

# EXTENDED regime only (all essays >= 778 tokens)
WINDOWS = [32, 64, 128, 256, 384, 512]
BURN_IN = 512
MAX_SCORE_TOKENS = 256
MIN_TOKENS = 778  # All essays must be >= this

# GPU settings
USE_4BIT = True  # True for T4, False for A100

print(f"Experiment: {EXPERIMENT}")
print(f"Cohort: {COHORT_PATH}")
print(f"\nEXTENDED regime only:")
print(f"  Windows: {WINDOWS}")
print(f"  Burn-in: {BURN_IN}")
print(f"  Min tokens: {MIN_TOKENS}")
print(f"\nPrimary DV: half_life (full, using all windows)")
print(f"Secondary DVs: delta_max, delta_128, delta_late")

## 1. Verify Cohort

In [ ]:
# Load and verify cohort
def load_cohort(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

cohort = load_cohort(COHORT_PATH)
df_cohort = pd.DataFrame(cohort)

print("="*70)
print("COHORT VERIFICATION")
print("="*70)

# Check 1: Total count
print(f"\n1. Total essays: {len(cohort)}")
assert len(cohort) == 150, f"Expected 150, got {len(cohort)}"
print("   PASS: 150 essays")

# Check 2: Score bin distribution
print(f"\n2. Score bin distribution:")
score_counts = df_cohort['score_bin'].value_counts()
for bin_name in ['low', 'mid', 'high']:
    n = score_counts.get(bin_name, 0)
    status = "PASS" if n == 50 else "FAIL"
    print(f"   {bin_name}: {n} [{status}]")
assert set(df_cohort['score_bin'].unique()) == {'low', 'mid', 'high'}
assert all(score_counts[b] == 50 for b in ['low', 'mid', 'high'])

# Check 3: Token counts >= MIN_TOKENS
print(f"\n3. Token count range:")
min_tok = df_cohort['token_count'].min()
max_tok = df_cohort['token_count'].max()
print(f"   Min: {min_tok}, Max: {max_tok}")
assert min_tok >= MIN_TOKENS, f"Min token count {min_tok} < {MIN_TOKENS}"
print(f"   PASS: All essays >= {MIN_TOKENS} tokens")

# Summary by score_bin
print(f"\n4. Token counts by score_bin:")
for bin_name in ['low', 'mid', 'high']:
    subset = df_cohort[df_cohort['score_bin'] == bin_name]
    print(f"   {bin_name}: median={subset['token_count'].median():.0f}, range=[{subset['token_count'].min()}, {subset['token_count'].max()}]")

print("\n" + "="*70)
print("COHORT VERIFICATION COMPLETE - ALL CHECKS PASSED")
print("="*70)

## 2. Load Model

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}")

if USE_4BIT:
    print("  Using 4-bit quantization (T4 mode)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    print("  Using float16 (A100 mode)")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

## 3. Core Functions

In [ ]:
@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    """Compute perplexity on tokens[target_start:target_end]."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return float('inf'), 0
    
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        token_loss = -log_probs[token_ids[i + 1]].item()
        total_loss += token_loss
        count += 1
    
    return math.exp(total_loss / count) if count > 0 else float('inf'), count


def compute_memory_curve(token_ids, windows, burn_in, max_score_tokens):
    """Compute memory curve from pre-tokenized input."""
    result = {'ppl_by_W': {}, 'token_count': len(token_ids)}
    n_tokens = len(token_ids)
    
    if n_tokens <= burn_in:
        return result
    
    target_end = min(n_tokens, burn_in + max_score_tokens)
    
    for W in windows:
        context_start = max(0, burn_in - W)
        actual_context = burn_in - context_start
        
        if actual_context < 4:
            continue
        
        truncated = token_ids[context_start:target_end]
        ppl, _ = compute_perplexity_on_region(truncated, actual_context, len(truncated))
        
        if not math.isinf(ppl):
            result['ppl_by_W'][W] = ppl
    
    return result


def compute_half_life(ppl_by_W, percentile=0.5):
    """Compute half-life: context length to achieve percentile of total benefit."""
    if len(ppl_by_W) < 2:
        return float('nan')
    
    items = sorted(ppl_by_W.items())
    windows = np.array([x[0] for x in items])
    ppls = np.array([x[1] for x in items])
    
    total_benefit = ppls[0] - ppls[-1]
    if total_benefit <= 0:
        return float('nan')
    
    target_ppl = ppls[0] - percentile * total_benefit
    
    for i in range(len(ppls) - 1):
        if ppls[i] >= target_ppl >= ppls[i + 1]:
            frac = (ppls[i] - target_ppl) / (ppls[i] - ppls[i + 1])
            return windows[i] + frac * (windows[i + 1] - windows[i])
    
    return windows[-1]


def compute_features(ppl_by_W):
    """Extract features from memory curve."""
    features = {
        'half_life': float('nan'),
        'delta_max': float('nan'),
        'delta_128': float('nan'),
        'delta_late': float('nan'),
    }
    
    if len(ppl_by_W) < 2:
        return features
    
    items = sorted(ppl_by_W.items())
    
    # delta_max: ppl_32 - ppl_512
    features['delta_max'] = items[0][1] - items[-1][1]
    
    # half_life: full windows
    features['half_life'] = compute_half_life(ppl_by_W, 0.5)
    
    # delta_128: ppl_32 - ppl_128
    ppl_32 = ppl_by_W.get(32)
    ppl_128 = ppl_by_W.get(128)
    if ppl_32 is not None and ppl_128 is not None:
        features['delta_128'] = ppl_32 - ppl_128
    
    # delta_late: delta_max - delta_128 (benefit from windows > 128)
    if not np.isnan(features['delta_max']) and not np.isnan(features['delta_128']):
        features['delta_late'] = features['delta_max'] - features['delta_128']
    
    return features

print("Functions defined")

## 4. Process Essays

In [ ]:
# Process all essays
results = []
start_time = time.time()

for essay in tqdm(cohort, desc="Processing"):
    token_ids = tokenizer.encode(essay['text'], add_special_tokens=False)
    n_tokens = len(token_ids)
    
    # All essays should be EXTENDED
    assert n_tokens >= MIN_TOKENS, f"Essay {essay['essay_id']} has {n_tokens} tokens < {MIN_TOKENS}"
    
    curve = compute_memory_curve(token_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
    features = compute_features(curve['ppl_by_W'])
    
    result = {
        'essay_id': essay.get('essay_id'),
        'score': essay.get('score'),
        'score_bin': essay.get('score_bin'),
        'grade': essay.get('grade'),
        'ell': essay.get('ell'),
        'prompt_id': essay.get('prompt_id'),
        'word_count': essay.get('word_count'),
        'token_count': n_tokens,
        'regime': 'extended',
        **features,
    }
    
    for W in WINDOWS:
        result[f'ppl_W{W}'] = curve['ppl_by_W'].get(W, float('nan'))
    
    results.append(result)

elapsed = time.time() - start_time
print(f"\nProcessed {len(results)} essays in {elapsed:.1f}s ({elapsed/len(results):.2f}s/essay)")

In [ ]:
# Create DataFrame and verify coverage
df = pd.DataFrame(results)

print("="*70)
print("COVERAGE VERIFICATION")
print("="*70)

# Check regime
print(f"\nRegime distribution:")
print(df['regime'].value_counts())
assert (df['regime'] == 'extended').all(), "Not all essays are EXTENDED!"
print("PASS: All essays in EXTENDED regime")

# Check ppl_W512 coverage
n_valid_512 = df['ppl_W512'].notna().sum()
print(f"\nppl_W512 coverage: {n_valid_512}/{len(df)} ({100*n_valid_512/len(df):.1f}%)")
assert n_valid_512 == len(df), f"Missing ppl_W512 for {len(df) - n_valid_512} essays"
print("PASS: All essays have valid ppl_W512")

# Window coverage table
print(f"\nWindow coverage by score_bin:")
print(f"{'score_bin':<10}", end="")
for W in WINDOWS:
    print(f"{'W='+str(W):>10}", end="")
print()
print("-"*70)
for sb in ['low', 'mid', 'high']:
    subset = df[df['score_bin'] == sb]
    print(f"{sb:<10}", end="")
    for W in WINDOWS:
        n = subset[f'ppl_W{W}'].notna().sum()
        print(f"{n:>10}", end="")
    print()

## 5. Descriptive Statistics

In [ ]:
# Group summary
GROUP_ORDER = ['low', 'mid', 'high']

print("="*70)
print("GROUP SUMMARY (score_bin)")
print("="*70)

summary_rows = []
for group in GROUP_ORDER:
    g_df = df[df['score_bin'] == group]
    row = {
        'group': group,
        'n': len(g_df),
        'token_count_median': g_df['token_count'].median(),
        'half_life_mean': g_df['half_life'].mean(),
        'half_life_sem': g_df['half_life'].sem(),
        'half_life_ci95': 1.96 * g_df['half_life'].sem(),
        'delta_max_mean': g_df['delta_max'].mean(),
        'delta_max_sem': g_df['delta_max'].sem(),
        'delta_128_mean': g_df['delta_128'].mean(),
        'delta_late_mean': g_df['delta_late'].mean(),
    }
    summary_rows.append(row)
    
    print(f"\n{group}: n={row['n']}, median_tokens={row['token_count_median']:.0f}")
    print(f"  PRIMARY:")
    print(f"    half_life:  {row['half_life_mean']:.1f} +/- {row['half_life_ci95']:.1f} (95% CI)")
    print(f"    delta_max:  {row['delta_max_mean']:.3f} +/- {1.96*row['delta_max_sem']:.3f}")
    print(f"  SECONDARY:")
    print(f"    delta_128:  {row['delta_128_mean']:.3f}")
    print(f"    delta_late: {row['delta_late_mean']:.3f}")

df_summary = pd.DataFrame(summary_rows)
df_summary

## 6. Statistical Tests

### A) No covariates: ANOVA + pairwise with Holm correction

In [ ]:
from scipy.stats import f_oneway
from statsmodels.stats.multitest import multipletests

print("="*70)
print("STATISTICAL TESTS - NO COVARIATES")
print("="*70)

# Get groups
low = df[df['score_bin'] == 'low']
mid = df[df['score_bin'] == 'mid']
high = df[df['score_bin'] == 'high']

# ============================================================
# ANOVA: half_life ~ C(score_bin)
# ============================================================
print("\n--- ANOVA: half_life ~ score_bin ---")
f_stat, p_val = f_oneway(low['half_life'], mid['half_life'], high['half_life'])
sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
print(f"F = {f_stat:.3f}, p = {p_val:.4f} {sig}")

# Pairwise t-tests with Holm correction
print("\nPairwise t-tests (Holm-corrected):")
pairs = [('low', 'mid'), ('mid', 'high'), ('low', 'high')]
raw_pvals_hl = []
t_stats_hl = []
for g1, g2 in pairs:
    v1 = df[df['score_bin'] == g1]['half_life']
    v2 = df[df['score_bin'] == g2]['half_life']
    t, p = stats.ttest_ind(v1, v2)
    raw_pvals_hl.append(p)
    t_stats_hl.append(t)

reject_hl, corrected_hl, _, _ = multipletests(raw_pvals_hl, method='holm')
for i, (g1, g2) in enumerate(pairs):
    sig = "***" if corrected_hl[i] < 0.001 else "**" if corrected_hl[i] < 0.01 else "*" if corrected_hl[i] < 0.05 else ""
    print(f"  {g1} vs {g2}: t={t_stats_hl[i]:.2f}, p_raw={raw_pvals_hl[i]:.4f}, p_holm={corrected_hl[i]:.4f} {sig}")

# ============================================================
# ANOVA: delta_max ~ C(score_bin)
# ============================================================
print("\n--- ANOVA: delta_max ~ score_bin ---")
f_stat, p_val = f_oneway(low['delta_max'], mid['delta_max'], high['delta_max'])
sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
print(f"F = {f_stat:.3f}, p = {p_val:.4f} {sig}")

# Pairwise t-tests with Holm correction
print("\nPairwise t-tests (Holm-corrected):")
raw_pvals_dm = []
t_stats_dm = []
for g1, g2 in pairs:
    v1 = df[df['score_bin'] == g1]['delta_max']
    v2 = df[df['score_bin'] == g2]['delta_max']
    t, p = stats.ttest_ind(v1, v2)
    raw_pvals_dm.append(p)
    t_stats_dm.append(t)

reject_dm, corrected_dm, _, _ = multipletests(raw_pvals_dm, method='holm')
for i, (g1, g2) in enumerate(pairs):
    sig = "***" if corrected_dm[i] < 0.001 else "**" if corrected_dm[i] < 0.01 else "*" if corrected_dm[i] < 0.05 else ""
    print(f"  {g1} vs {g2}: t={t_stats_dm[i]:.2f}, p_raw={raw_pvals_dm[i]:.4f}, p_holm={corrected_dm[i]:.4f} {sig}")

### B) With length covariate (OLS)

In [ ]:
print("="*70)
print("STATISTICAL TESTS - WITH LENGTH COVARIATE")
print("="*70)

# Prepare data
df_reg = df.copy()
df_reg['score_bin'] = pd.Categorical(df_reg['score_bin'], categories=['low', 'mid', 'high'], ordered=True)

# Standardize token_count
df_reg['token_count_z'] = (df_reg['token_count'] - df_reg['token_count'].mean()) / df_reg['token_count'].std()

# ============================================================
# OLS: half_life ~ C(score_bin) + token_count_z
# ============================================================
print("\n--- OLS: half_life ~ C(score_bin) + token_count_z ---")
ols_hl = smf.ols('half_life ~ C(score_bin) + token_count_z', data=df_reg).fit()
print(ols_hl.summary().tables[1])
print(f"\nR-squared: {ols_hl.rsquared:.4f}")

# Standardized beta for token_count_z
y_std = df_reg['half_life'].std()
beta_token_z = ols_hl.params['token_count_z']
std_beta_token = beta_token_z * df_reg['token_count_z'].std() / y_std
print(f"\nStandardized beta (token_count_z): {std_beta_token:.3f}")

# ============================================================
# OLS: delta_max ~ C(score_bin) + token_count_z
# ============================================================
print("\n" + "-"*70)
print("\n--- OLS: delta_max ~ C(score_bin) + token_count_z ---")
ols_dm = smf.ols('delta_max ~ C(score_bin) + token_count_z', data=df_reg).fit()
print(ols_dm.summary().tables[1])
print(f"\nR-squared: {ols_dm.rsquared:.4f}")

# Standardized beta for token_count_z
y_std = df_reg['delta_max'].std()
beta_token_z = ols_dm.params['token_count_z']
std_beta_token = beta_token_z * df_reg['token_count_z'].std() / y_std
print(f"\nStandardized beta (token_count_z): {std_beta_token:.3f}")

## 7. Plots

In [ ]:
# Color scheme
COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}

# Plot 1: Memory curves (means + 95% CI)
fig1, ax1 = plt.subplots(figsize=(10, 6))

for group in GROUP_ORDER:
    g_df = df[df['score_bin'] == group]
    means = [g_df[f'ppl_W{W}'].mean() for W in WINDOWS]
    ci95 = [1.96 * g_df[f'ppl_W{W}'].sem() for W in WINDOWS]
    ax1.errorbar(WINDOWS, means, yerr=ci95, marker='o', capsize=4,
                label=f"{group} (n={len(g_df)})", color=COLORS[group], linewidth=2, markersize=8)

ax1.set_xlabel('Context Window (tokens)', fontsize=12)
ax1.set_ylabel('Perplexity', fontsize=12)
ax1.set_title('Memory Curves by Score Bin (score_long cohort)\nAll windows [32-512], 95% CI', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(WINDOWS)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Box/violin plots of half_life and delta_max
fig2, axes = plt.subplots(1, 2, figsize=(12, 5))

# half_life
ax = axes[0]
sns.violinplot(data=df, x='score_bin', y='half_life', order=GROUP_ORDER,
               palette=COLORS, ax=ax, inner='box')
ax.set_xlabel('Score Bin', fontsize=12)
ax.set_ylabel('Half-life (tokens)', fontsize=12)
ax.set_title('Half-life by Score Bin', fontsize=14)

# delta_max
ax = axes[1]
sns.violinplot(data=df, x='score_bin', y='delta_max', order=GROUP_ORDER,
               palette=COLORS, ax=ax, inner='box')
ax.set_xlabel('Score Bin', fontsize=12)
ax.set_ylabel('Delta Max (perplexity reduction)', fontsize=12)
ax.set_title('Delta Max by Score Bin', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3 & 4: Scatter plots vs token_count
fig3, axes = plt.subplots(1, 2, figsize=(14, 5))

# half_life vs token_count
ax = axes[0]
for group in GROUP_ORDER:
    g_df = df[df['score_bin'] == group]
    ax.scatter(g_df['token_count'], g_df['half_life'], 
               c=COLORS[group], label=group, alpha=0.7, s=50)
ax.set_xlabel('Token Count', fontsize=12)
ax.set_ylabel('Half-life (tokens)', fontsize=12)
ax.set_title('Half-life vs Token Count', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

# Add regression line
z = np.polyfit(df['token_count'], df['half_life'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['token_count'].min(), df['token_count'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', alpha=0.5, label='Overall trend')

# delta_max vs token_count
ax = axes[1]
for group in GROUP_ORDER:
    g_df = df[df['score_bin'] == group]
    ax.scatter(g_df['token_count'], g_df['delta_max'], 
               c=COLORS[group], label=group, alpha=0.7, s=50)
ax.set_xlabel('Token Count', fontsize=12)
ax.set_ylabel('Delta Max', fontsize=12)
ax.set_title('Delta Max vs Token Count', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

# Add regression line
z = np.polyfit(df['token_count'], df['delta_max'], 1)
p = np.poly1d(z)
ax.plot(x_line, p(x_line), 'k--', alpha=0.5, label='Overall trend')

plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
# Save all results
output_dir = Path(f"{OUTPUT_BASE}/{EXPERIMENT}")
output_dir.mkdir(parents=True, exist_ok=True)

# Essay-level results
df.to_csv(output_dir / 'essay_level_results.csv', index=False)

# Group summary
df_summary.to_csv(output_dir / 'group_summary.csv', index=False)

# Regression summaries
with open(output_dir / 'regression_summary_half_life.txt', 'w') as f:
    f.write("OLS: half_life ~ C(score_bin) + token_count_z\n\n")
    f.write(ols_hl.summary().as_text())

with open(output_dir / 'regression_summary_delta_max.txt', 'w') as f:
    f.write("OLS: delta_max ~ C(score_bin) + token_count_z\n\n")
    f.write(ols_dm.summary().as_text())

# Save figures
fig1.savefig(output_dir / 'memory_curves.png', dpi=150, bbox_inches='tight')
fig2.savefig(output_dir / 'violin_plots.png', dpi=150, bbox_inches='tight')
fig3.savefig(output_dir / 'scatter_plots.png', dpi=150, bbox_inches='tight')

print(f"Saved to {output_dir}/")
print(f"  - essay_level_results.csv ({len(df)} essays)")
print(f"  - group_summary.csv")
print(f"  - regression_summary_half_life.txt")
print(f"  - regression_summary_delta_max.txt")
print(f"  - memory_curves.png")
print(f"  - violin_plots.png")
print(f"  - scatter_plots.png")